In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db")
vs_index

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [5]:
import os
import sys
sys.path.append("../Week_01_Agentic_RAG")
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [6]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client,
)

In [7]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [8]:
vs_index.close()